In [1]:
import pandas as pd 
import geopandas as gpd

In [2]:
import os
os.chdir('/users/ctlin/fooddesertproject')

In [5]:
csv_paths = [
    "health_data/composite_data/Bexar_health_composite_score.csv",
    "health_data/composite_data/Harris_health_composite_score.csv",
    "health_data/composite_data/Dallas_health_composite_score.csv",
    "health_data/composite_data/Travis_health_composite_score.csv",
    "health_data/composite_data/Tarrant_health_composite_score.csv",
]

In [6]:
dfs = [pd.read_csv(p) for p in csv_paths]
all_counties = pd.concat(dfs, ignore_index=True)

In [7]:
# keep only the final health composite score 
all_counties = all_counties[["TractFIPS", "final_health_composite"]]
all_counties["TractFIPS"] = all_counties["TractFIPS"].astype(str).str.zfill(11)

In [10]:
gdf = gpd.read_file("modeling_data/modeling_data_final.gpkg")
print(gdf.columns)

Index(['COUNTY', 'GEOID', 'tract', 'walking_ind', 'walkind_inv_perc',
       'total_pop', 'total_pop_moe', 'below_200_fed_poverty_percentage',
       'no_hs_diploma', 'uninsured', 'E_CHD', 'E_DIABETES', 'E_AFAM',
       'E_ASIAN', 'E_HISP', 'median_age', 'ALAND', 'AWATER',
       'distance_to_nearest_grocery', 'distance_to_nearest_fm',
       'distance_to_nearest_uf', 'median_income', 'pct_no_vehicle',
       'geometry'],
      dtype='object')


In [11]:
gdf["GEOID"] = gdf["GEOID"].astype(str).str.zfill(11)

In [12]:
# Merge (left join keeps all geometries even if a tract has no score)
merged = gdf.merge(
    all_counties,
    left_on="GEOID",
    right_on="TractFIPS",
    how="left"
)

In [13]:
merged.to_file("complete_modeling_data.gpkg", driver="GPKG")